# Proyecto Fin de Curso: Predicción de la Demanda Energética en España (2015-2018)

## HITO 2

**Estudiante:** Marta Requejo Merino 

**Especialidad:** Grado en IA y Big Data 

---

In [1]:
# System and Environment
import os
from dotenv import load_dotenv

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Scikit-Learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Machine Learning - XGBoost
import xgboost as xgb

# Persistence
import joblib

import holidays


In [2]:
# Definir rutas basadas en la estructura
BASE_DIR = os.path.dirname(os.getcwd()) 
DATA_DIR = os.path.join(BASE_DIR, 'datos')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datos_procesados')
MODELS_DIR = os.path.join(BASE_DIR, 'modelos')

# Asegurar que existan las carpetas
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Ruta base configurada: {BASE_DIR}")

Ruta base configurada: c:\Users\marti\Desktop\proyecto energia


In [3]:
# Carga de datos preprocesados
df = pd.read_csv(os.path.join(PROCESSED_DIR, "datos_limpios_hito01.csv"))

# Ajuste de zona horaria local (Europa/Madrid)
df['time_local'] = pd.to_datetime(df['time_local'], utc=True).dt.tz_convert('Europe/Madrid')

# Split cronológico (80% Train, 20% Test)
indice_corte = int(len(df) * 0.8)
train_df = df.iloc[:indice_corte].copy()
test_df = df.iloc[indice_corte:].copy()

# Definición de features (X) y target (y)
features = ['hour', 'month', 'day_of_week', 'temp_madrid', 'temp_barcelona', 'temp_seville',
            'lag_24h', 'lag_168h', 'rolling_mean_24h']
target = 'total_load_actual'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# Carga del escalador maestro
scaler = joblib.load(os.path.join(MODELS_DIR, "scaler_maestro.pkl"))

# Escalado de características (solo transform)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

print("Datos listos para entrenar")

Datos listos para entrenar


## 6. Creación, Entrenamiento y Validación de Modelos

El objetivo es entrenar algoritmos para que aprendan la relación entre nuestros predictores y la demanda real. Cumpliendo con los requisitos del proyecto, compararemos dos modelos distintos, modificaremos sus parámetros mediante validación cruzada y evaluaremos su bondad de ajuste (MAE, RMSE, R²).

### 6.1. Definición de la Función de Evaluación

In [4]:
def evaluar_modelo(y_true, y_pred, nombre_modelo):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"=========================================")
    print(f"RESULTADOS: {nombre_modelo.upper()}")
    print(f"=========================================")
    print(f" -> MAE  : {mae:.2f} MW")
    print(f" -> RMSE : {rmse:.2f} MW")
    print(f" -> R²   : {r2:.4f} ({(r2*100):.2f}%)")
    print(f"=========================================\n")
    return mae, rmse, r2

### 6.2 Modelos

**Justificación de la selección:**
Para esta primera fase de entrenamiento, hemos seleccionado tres algoritmos de distinta naturaleza.
1. **Regresión Ridge (Línea Base Lineal):** Es una evolución de la Regresión Lineal clásica que incluye regularización para evitar el sobreajuste. La utilizamos como nuestro "modelo de control". Dado que la relación entre clima y demanda no es lineal (tiene forma de "U"), esperamos que este modelo alcance un techo, demostrando la necesidad de algoritmos más complejos.
2. **Random Forest Regressor:** Crea cientos de árboles de decisión independientes. Es teóricamente ideal para captar relaciones no lineales sin necesidad de complejas transformaciones trigonométricas.
3. **XGBoost (Extreme Gradient Boosting):**"Constituye uno de los modelos más avanzados y eficaces en la actualidad para el análisis predictivo con datos estructurado. Aprende secuencialmente de sus propios errores y está altamente optimizado.

#### 6.2.1. Entrenamiento Modelo 1: Regresión Ridge

In [5]:
print("Optimizando Regresión Ridge...")

# Búsqueda de hiperparámetros mediante validación cruzada
parametros_ridge = {'alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
grid_ridge = GridSearchCV(Ridge(), parametros_ridge, cv=3, scoring='r2')
grid_ridge.fit(X_train_scaled, y_train)

# Selección del modelo óptimo
mejor_ridge = grid_ridge.best_estimator_
print(f"Mejor parámetro seleccionado: {grid_ridge.best_params_}")

# Predicción y evaluación en el conjunto de prueba
y_pred_ridge = mejor_ridge.predict(X_test_scaled)
evaluar_modelo(y_test, y_pred_ridge, "Regresión Ridge")

Optimizando Regresión Ridge...
Mejor parámetro seleccionado: {'alpha': 100.0}
RESULTADOS: REGRESIÓN RIDGE
 -> MAE  : 2018.61 MW
 -> RMSE : 2693.92 MW
 -> R²   : 0.6445 (64.45%)



(2018.6075391885931, 2693.918846014506, 0.6445406997490176)

#### 6.2.2 Modelo 2: Random Forest Regressor

In [6]:
print("Optimizando Random Forest...")

# Búsqueda de hiperparámetros mediante validación cruzada
parametros_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15]
}
random_rf = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                               param_distributions=parametros_rf, n_iter=4, cv=3, random_state=42)
random_rf.fit(X_train_scaled, y_train)

# Selección del modelo óptimo
mejor_rf = random_rf.best_estimator_
print(f"Mejores parámetros seleccionados: {random_rf.best_params_}")

# Predicción y evaluación en el conjunto de prueba
y_pred_rf = mejor_rf.predict(X_test_scaled)
evaluar_modelo(y_test, y_pred_rf, "Random Forest (Baseline)")

Optimizando Random Forest...
Mejores parámetros seleccionados: {'n_estimators': 100, 'max_depth': 15}
RESULTADOS: RANDOM FOREST (BASELINE)
 -> MAE  : 1044.20 MW
 -> RMSE : 1528.17 MW
 -> R²   : 0.8856 (88.56%)



(1044.1998520038899, 1528.1697914045626, 0.8856163145771552)

#### 6.2.3 Modelo 4: XGBoost Regressorº

In [7]:
print("Optimizando XGBoost...")

# Búsqueda de hiperparámetros mediante validación cruzada
parametros_xgb = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7]
}
random_xgb = RandomizedSearchCV(xgb.XGBRegressor(random_state=42, n_jobs=-1),
                               param_distributions=parametros_xgb, n_iter=4, cv=3, random_state=42)
random_xgb.fit(X_train_scaled, y_train)

# Selección del modelo óptimo
mejor_xgb = random_xgb.best_estimator_
print(f"Mejores parámetros seleccionados: {random_xgb.best_params_}")

# Predicción y evaluación en el conjunto de prueba
y_pred_xgb = mejor_xgb.predict(X_test_scaled)
evaluar_modelo(y_test, y_pred_xgb, "XGBoost (Baseline)")

Optimizando XGBoost...
Mejores parámetros seleccionados: {'n_estimators': 150, 'max_depth': 7, 'learning_rate': 0.1}
RESULTADOS: XGBOOST (BASELINE)
 -> MAE  : 1022.70 MW
 -> RMSE : 1495.42 MW
 -> R²   : 0.8905 (89.05%)



(1022.7016285368911, 1495.415746626936, 0.8904670553073539)

### 6.3. Evaluación Comparativa y Selección de Modelos

Vemos que modelos son los más óptimos antes de continuar la optmización para comprobar si alguno se queda demasiado atrás.

In [8]:
# Agrupación de los modelos base entrenados
modelos_base = {
    "Regresión Ridge": mejor_ridge,
    "Random Forest": mejor_rf,
    "XGBoost": mejor_xgb
}

lista_resultados = []

# Evaluación y extracción de métricas
for nombre, modelo in modelos_base.items():
    y_pred = modelo.predict(X_test_scaled)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    lista_resultados.append({
        "Modelo": nombre,
        "R² (%)": round(r2 * 100, 2),
        "MAE (MW)": round(mae, 2),
        "RMSE (MW)": round(rmse, 2)
    })

# Consolidación y ordenación de los resultados
df_comparativa_base = pd.DataFrame(lista_resultados)
df_comparativa_base = df_comparativa_base.sort_values(by="R² (%)", ascending=False).reset_index(drop=True)
display(df_comparativa_base)

,Modelo,R² (%),MAE (MW),RMSE (MW)
0,XGBoost,89.05,1022.70,1495.42
1,Random Forest,88.56,1044.20,1528.17
2,Regresión Ridge,64.45,2018.61,2693.92


**Conclusiones de la Evaluación Base:**

Al observar la tabla comparativa, la arquitectura matemática de los modelos explica claramente sus resultados:

* **El Límite del Modelo Lineal:** La Regresión Ridge se posiciona en último lugar. Aunque es un modelo rápido y robusto, su naturaleza lineal le impide comprender matemáticamente que tanto a los 5ºC como a los 35ºC el consumo eléctrico sube (forma de "U"). Para la Regresión, la línea solo puede ir en una dirección, lo que limita su capacidad predictiva y aumenta su margen de error (MAE).
* **La Superioridad de los Árboles:** Por el contrario, XGBoost y Random Forest superan ampliamente el 88% de varianza $R^2$. Al estar basados en árboles de decisión, pueden hacer divisiones lógicas, captando a la perfección la estacionalidad climática.

Habiendo validado que una aproximación puramente lineal no es suficiente para resolver la complejidad de la demanda eléctrica, **descartamos la Regresión Ridge**. Para la siguiente fase de experimentación avanzada, nos quedaremos exclusivamente con nuestros dos modelos campeones: **XGBoost y Random Forest**.

### 6.4. Optimización y creación de nuevas variables

Como ya hemos visto que XGBoost es el modelo que mejor funciona (con un $R^2$ inicial del **89.05%**), ahora vamos a intentar mejorar esa nota creando variables nuevas.

Para evitar confundir al algoritmo metiéndole demasiados datos de golpe, vamos a ir probando estas nuevas variables una a una:
1. Entrenamos el modelo añadiendo la nueva variable.
2. Si el porcentaje de acierto ($R^2$) mejora y el error (MAE) baja, **nos quedamos con la variable**.
3. Si el resultado empeora o se queda igual, **la descartamos**.

Vamos a añadir variables para **días festivos, fines de semana, horario laboral yº temperaturas extremas.**

In [9]:
# Inicialización del conjunto de variables predictoras base
variables_ganadoras = features.copy()
record_r2 = 0.8905  # R² de referencia del modelo XGBoost

print(f"R² de referencia a superar: {record_r2 * 100:.2f}%")

# Generación de variables experimentales en los conjuntos de entrenamiento y prueba

# 1. Festivos nacionales
festivos = holidays.Spain(years=[2015, 2016, 2017, 2018])
train_df['is_holiday'] = train_df['time_local'].dt.date.apply(lambda x: 1 if x in festivos else 0)
test_df['is_holiday'] = test_df['time_local'].dt.date.apply(lambda x: 1 if x in festivos else 0)

# 2. Fines de semana
train_df['is_weekend'] = train_df['time_local'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
test_df['is_weekend'] = test_df['time_local'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# 3. Horario laboral (08:00 - 18:00)
train_df['is_business_hour'] = train_df['time_local'].dt.hour.apply(lambda x: 1 if 8 <= x <= 18 else 0)
test_df['is_business_hour'] = test_df['time_local'].dt.hour.apply(lambda x: 1 if 8 <= x <= 18 else 0)

# 4. Temperaturas extremas (>32ºC o <5ºC)
train_df['is_extreme_temp'] = train_df['temp_madrid'].apply(lambda x: 1 if x > 305.15 or x < 278.15 else 0)
test_df['is_extreme_temp'] = test_df['temp_madrid'].apply(lambda x: 1 if x > 305.15 or x < 278.15 else 0)

print("Variables experimentales generadas con éxito.")

R² de referencia a superar: 89.05%
Variables experimentales generadas con éxito.


#### 6.4.1 Festivos

In [10]:
print("--- EXPERIMENTO 1: AÑADIENDO FESTIVOS ---")
features_prueba = variables_ganadoras + ['is_holiday']

# Escalado de variables para el modelo
scaler_prueba = StandardScaler()
X_train_pr = pd.DataFrame(scaler_prueba.fit_transform(train_df[features_prueba]), columns=features_prueba)
X_test_pr = pd.DataFrame(scaler_prueba.transform(test_df[features_prueba]), columns=features_prueba)

# Entrenamiento y evaluación del modelo
random_xgb.fit(X_train_pr, y_train)
y_pred_pr = random_xgb.best_estimator_.predict(X_test_pr)
mae, rmse, r2 = evaluar_modelo(y_test, y_pred_pr, "XGBoost (+ Festivos)")

# Validación de rendimiento e integración de la variable
if r2 > record_r2:
    print("Mejora de rendimiento detectada. Variable 'is_holiday' incorporada al modelo final.")
    variables_ganadoras.append('is_holiday')
    record_r2 = r2
else:
    print("Rendimiento inferior o estancado. Variable 'is_holiday' descartada.")

--- EXPERIMENTO 1: AÑADIENDO FESTIVOS ---
RESULTADOS: XGBOOST (+ FESTIVOS)
 -> MAE  : 1014.24 MW
 -> RMSE : 1484.70 MW
 -> R²   : 0.8920 (89.20%)

Mejora de rendimiento detectada. Variable 'is_holiday' incorporada al modelo final.


#### 6.4.2 Fin de semana

In [11]:
print("--- EXPERIMENTO 2: AÑADIENDO FIN DE SEMANA ---")
features_prueba = variables_ganadoras + ['is_weekend']

# Escalado de variables para el modelo
scaler_prueba = StandardScaler()
X_train_pr = pd.DataFrame(scaler_prueba.fit_transform(train_df[features_prueba]), columns=features_prueba)
X_test_pr = pd.DataFrame(scaler_prueba.transform(test_df[features_prueba]), columns=features_prueba)

# Entrenamiento y evaluación del modelo
random_xgb.fit(X_train_pr, y_train)
y_pred_pr = random_xgb.best_estimator_.predict(X_test_pr)
mae, rmse, r2 = evaluar_modelo(y_test, y_pred_pr, "XGBoost (+ Fin de Semana)")

# Validación de rendimiento e integración de la variable
if r2 > record_r2:
    print("Mejora de rendimiento detectada. Variable 'is_weekend' incorporada al modelo final.")
    variables_ganadoras.append('is_weekend')
    record_r2 = r2
else:
    print("Rendimiento inferior o estancado. Variable 'is_weekend' descartada por redundancia.")

--- EXPERIMENTO 2: AÑADIENDO FIN DE SEMANA ---
RESULTADOS: XGBOOST (+ FIN DE SEMANA)
 -> MAE  : 1014.24 MW
 -> RMSE : 1484.70 MW
 -> R²   : 0.8920 (89.20%)

Rendimiento inferior o estancado. Variable 'is_weekend' descartada por redundancia.


#### 6.4.3 Horario laboral

In [12]:
print("--- EXPERIMENTO 3: AÑADIENDO HORARIO LABORAL ---")
features_prueba = variables_ganadoras + ['is_business_hour']

# Escalado de variables para el modelo
scaler_prueba = StandardScaler()
X_train_pr = pd.DataFrame(scaler_prueba.fit_transform(train_df[features_prueba]), columns=features_prueba)
X_test_pr = pd.DataFrame(scaler_prueba.transform(test_df[features_prueba]), columns=features_prueba)

# Entrenamiento y evaluación del modelo
random_xgb.fit(X_train_pr, y_train)
y_pred_pr = random_xgb.best_estimator_.predict(X_test_pr)
mae, rmse, r2 = evaluar_modelo(y_test, y_pred_pr, "XGBoost (+ Horario Laboral)")

# Validación de rendimiento e integración de la variable
if r2 > record_r2:
    print("Mejora de rendimiento detectada. Variable 'is_business_hour' incorporada al modelo final.")
    variables_ganadoras.append('is_business_hour')
    record_r2 = r2
else:
    print("Rendimiento inferior o estancado. Variable 'is_business_hour' descartada por redundancia.")

--- EXPERIMENTO 3: AÑADIENDO HORARIO LABORAL ---
RESULTADOS: XGBOOST (+ HORARIO LABORAL)
 -> MAE  : 1017.23 MW
 -> RMSE : 1488.06 MW
 -> R²   : 0.8915 (89.15%)

Rendimiento inferior o estancado. Variable 'is_business_hour' descartada por redundancia.


#### 6.4.4 Clima extremo

In [13]:
print("--- EXPERIMENTO 4: AÑADIENDO CLIMA EXTREMO ---")
features_prueba = variables_ganadoras + ['is_extreme_temp']

# Escalado de variables para el modelo
scaler_prueba = StandardScaler()
X_train_pr = pd.DataFrame(scaler_prueba.fit_transform(train_df[features_prueba]), columns=features_prueba)
X_test_pr = pd.DataFrame(scaler_prueba.transform(test_df[features_prueba]), columns=features_prueba)

# Entrenamiento y evaluación del modelo
random_xgb.fit(X_train_pr, y_train)
y_pred_pr = random_xgb.best_estimator_.predict(X_test_pr)
mae, rmse, r2 = evaluar_modelo(y_test, y_pred_pr, "XGBoost (+ Clima Extremo)")

# Validación de rendimiento e integración de la variable
if r2 > record_r2:
    print("Mejora de rendimiento detectada. Variable 'is_extreme_temp' incorporada al modelo final.")
    variables_ganadoras.append('is_extreme_temp')
    record_r2 = r2
else:
    print("Rendimiento inferior o estancado. Variable 'is_extreme_temp' descartada.")

--- EXPERIMENTO 4: AÑADIENDO CLIMA EXTREMO ---
RESULTADOS: XGBOOST (+ CLIMA EXTREMO)
 -> MAE  : 1013.80 MW
 -> RMSE : 1483.39 MW
 -> R²   : 0.8922 (89.22%)

Mejora de rendimiento detectada. Variable 'is_extreme_temp' incorporada al modelo final.


### 6.5 La importancia de los Lags

Para demostrar que usar datos del pasado es algo necesario para que el modelo funcione mejor vamor a proceder a las estas columnas para comprobar el rendimiento del modelo sin estas.
Hemos quitado todas las columnas que miran al pasado (lag_24h, lag_168h y rolling_mean_24h) y hemos obligado a nuestro mejor modelo (XGBoost) a adivinar el consumo usando únicamente los datos de ahora: el calendario, la hora y la temperatura.

In [14]:
print("--- EXPERIMENTO: XGBOOST SIN MEMORIA HISTÓRICA ---")

# Definición de variables excluyendo la memoria
features_sin_lags = [
    'hour',
    'month',
    'day_of_week',
    'temp_madrid',
    'temp_barcelona',
    'temp_seville',
    'is_holiday',
    'is_extreme_temp'
]

print(f"Entrenando modelo sin lags con {len(features_sin_lags)} variables...")

# Escalado de variables para el conjunto de datos reducido
scaler_ablacion = StandardScaler()
X_train_ablacion = pd.DataFrame(scaler_ablacion.fit_transform(train_df[features_sin_lags]), columns=features_sin_lags)
X_test_ablacion = pd.DataFrame(scaler_ablacion.transform(test_df[features_sin_lags]), columns=features_sin_lags)

# Entrenamiento y evaluación del modelo sin registros históricos
modelo_nolag = xgb.XGBRegressor(n_estimators=150, max_depth=7, learning_rate=0.1, random_state=42)
modelo_nolag.fit(X_train_ablacion, y_train)

# PREDECIMOS CON EL MODELO NUEVO (CORREGIDO)
y_pred_ablacion = modelo_nolag.predict(X_test_ablacion)

mae_ab, rmse_ab, r2_ab = evaluar_modelo(y_test, y_pred_ablacion, "XGBoost (Sin Lags)")

print(f"IMPACTO DEL ESTUDIO:")
print(f"Rendimiento resultante: {(r2_ab*100):.2f}% (Rendimiento base: 89.22%).")

--- EXPERIMENTO: XGBOOST SIN MEMORIA HISTÓRICA ---
Entrenando modelo sin lags con 8 variables...
RESULTADOS: XGBOOST (SIN LAGS)
 -> MAE  : 1862.08 MW
 -> RMSE : 2666.05 MW
 -> R²   : 0.6519 (65.19%)

IMPACTO DEL ESTUDIO:
Rendimiento resultante: 65.19% (Rendimiento base: 89.22%).


**Conclusiones:**

Al quitarle la memoria histórica, el rendimiento del modelo se desploma (pasamos de un $R^2$ del 89.22% a un 67.35%). Esto comprueba que el consumo eléctrico no depende solo de lo que pasa en el instante actual, sino que se basa mucho en la rutina y la inercia de días anteriores. Con esto demostramos que usar lags es fundamental para darle al modelo la misma información lógica que tendría cualquier trabajador de Red Eléctrica para hacer su trabajo.


## Guardado de datos

In [15]:
# 1. --- ENTRENAMIENTO: MODELO CON LAGS ---
scaler_final = StandardScaler()
X_train_final = pd.DataFrame(scaler_final.fit_transform(train_df[variables_ganadoras]), columns=variables_ganadoras)
modelo_definitivo = xgb.XGBRegressor(**random_xgb.best_params_, random_state=42, n_jobs=-1)
modelo_definitivo.fit(X_train_final, y_train)

# Persistencia de artefactos: Modelo, escalador y variables (Lags)
os.makedirs(MODELS_DIR, exist_ok=True)
modelo_definitivo.save_model(os.path.join(MODELS_DIR, "modelo_energia_es_lag.json"))
joblib.dump(scaler_final, os.path.join(MODELS_DIR, "escalador_energia_es_lag.pkl"))
joblib.dump(variables_ganadoras, os.path.join(MODELS_DIR, "variables_energia_es_lag.pkl"))

# 2. --- ENTRENAMIENTO: MODELO SIN LAGS (Baseline simplificado) ---
variables_sin_lags = [v for v in variables_ganadoras if 'lag' not in v and 'rolling' not in v]
scaler_no_lag = StandardScaler()
X_train_no_lag = pd.DataFrame(scaler_no_lag.fit_transform(train_df[variables_sin_lags]), columns=variables_sin_lags)
modelo_sin_lags = xgb.XGBRegressor(**random_xgb.best_params_, random_state=42, n_jobs=-1)
modelo_sin_lags.fit(X_train_no_lag, y_train)

# Persistencia de artefactos: Modelo y escalador (No-Lags)
modelo_sin_lags.save_model(os.path.join(MODELS_DIR, "modelo_energia_es_sin_lag.json"))
joblib.dump(scaler_no_lag, os.path.join(MODELS_DIR, "escalador_energia_es_sin_lag.pkl"))
joblib.dump(variables_sin_lags, os.path.join(MODELS_DIR, "variables_energia_es_sin_lag.pkl"))

# 3. --- EXPORTACIÓN DE DATOS ---
# Exportación del dataset completo para validación del Hito 2
test_df.to_csv(os.path.join(PROCESSED_DIR, "test_df_hito02.csv"), index=False)

# 4. --- EXPORTACIÓN PARA INFERENCIA (Hito 3) ---
columnas_necesarias = [
    'total_load_actual', 'hour', 'month', 'day_of_week', 
    'temp_madrid', 'temp_barcelona', 'temp_seville', 
    'is_holiday', 'is_extreme_temp', 
    'lag_24h', 'lag_168h', 'rolling_mean_24h'
]

# Filtrado y preparación del set de inferencia con marca temporal
test_export = test_df[columnas_necesarias].copy()
if 'time' in test_df.columns:
    test_export['fecha_hora'] = test_df['time']

# Exportación del subconjunto optimizado para el Hito 3
test_export.to_csv(os.path.join(PROCESSED_DIR, "datos_test_procesados.csv"), index=False)

print("¡Hito 2 completado! Modelos guardados en:", MODELS_DIR)
print("Datasets exportados en:", PROCESSED_DIR)

¡Hito 2 completado! Modelos guardados en: c:\Users\marti\Desktop\proyecto energia\modelos
Datasets exportados en: c:\Users\marti\Desktop\proyecto energia\datos_procesados
